In [5]:
import xarray as xr
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import os, re
import pandas as pd

In [6]:
os.chdir('F:\\geodata\\et_dataset_chenhuiling\\Daily\\')

In [7]:
file_list = [f"{y}\\ET.SiTHv2.A{y}.nc" for y in range(2000, 2021)]

In [8]:
basin_gdf = gpd.read_file(r'F:\geodata\river_runoff_obs\Tarim.shp')

In [9]:
ds = xr.open_dataset(file_list[0])
lon_arr = ds.variables['lon'].values
lat_arr = ds.variables['lat'].values
lon_grid, lat_grid = np.meshgrid(lon_arr, lat_arr)

# Create points and track indices
points = []
lon_indices = []
lat_indices = []

for lat_idx, lat in enumerate(lat_arr):
    for lon_idx, lon in enumerate(lon_arr):
        points.append(Point(lon, lat))  # Create Point object
        lon_indices.append(lon_idx)    # Track longitude index
        lat_indices.append(lat_idx)    # Track latitude index

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(geometry=points)
gdf['lon'] = gdf.geometry.x  # Extract longitude (X-coordinate)
gdf['lat'] = gdf.geometry.y  # Extract latitude (Y-coordinate)
gdf['lon_index'] = lon_indices  # Add longitude indices
gdf['lat_index'] = lat_indices  # Add latitude indices

# Check if CRS is set
if gdf.crs is None:
    # Set the CRS to WGS84 (latitude and longitude in degrees)
    gdf = gdf.set_crs("EPSG:4326")
    gdf = gdf.to_crs(basin_gdf.crs)

In [10]:
merged_gdf = gdf.sjoin(basin_gdf, how='left')
filtered_gdf = merged_gdf.dropna(subset=['abbre'])
filtered_df = filtered_gdf[[ 'lon', 'lat', 'lon_index', 'lat_index', 'abbre']].copy()
filtered_df

,lon,lat,lon_index,lat_index,abbre
1683849,84.949997,43.250000,2649,467,dsk
1683850,85.050003,43.250000,2650,467,dsk
1683851,85.150002,43.250000,2651,467,dsk
1683853,85.349998,43.250000,2653,467,dsk
1687439,83.949997,43.150002,2639,468,dsk
...,...,...,...,...,...
1989804,80.449997,34.750000,2604,552,wlwt
1989805,80.550003,34.750000,2605,552,wlwt
1989806,80.650002,34.750000,2606,552,wlwt
1989807,80.750000,34.750000,2607,552,wlwt


In [11]:
filtered_df =filtered_df.iloc[::20]

In [12]:
# Convert your DataFrame to lists for indexing
lon_indices = filtered_df['lon_index'].tolist()
lat_indices = filtered_df['lat_index'].tolist()

In [13]:
file_list.reverse()

In [14]:
file_list   

['2020\\ET.SiTHv2.A2020.nc',
 '2019\\ET.SiTHv2.A2019.nc',
 '2018\\ET.SiTHv2.A2018.nc',
 '2017\\ET.SiTHv2.A2017.nc',
 '2016\\ET.SiTHv2.A2016.nc',
 '2015\\ET.SiTHv2.A2015.nc',
 '2014\\ET.SiTHv2.A2014.nc',
 '2013\\ET.SiTHv2.A2013.nc',
 '2012\\ET.SiTHv2.A2012.nc',
 '2011\\ET.SiTHv2.A2011.nc',
 '2010\\ET.SiTHv2.A2010.nc',
 '2009\\ET.SiTHv2.A2009.nc',
 '2008\\ET.SiTHv2.A2008.nc',
 '2007\\ET.SiTHv2.A2007.nc',
 '2006\\ET.SiTHv2.A2006.nc',
 '2005\\ET.SiTHv2.A2005.nc',
 '2004\\ET.SiTHv2.A2004.nc',
 '2003\\ET.SiTHv2.A2003.nc',
 '2002\\ET.SiTHv2.A2002.nc',
 '2001\\ET.SiTHv2.A2001.nc',
 '2000\\ET.SiTHv2.A2000.nc']

In [15]:
for file_name in file_list: # here file_name is the path
    print(file_name)
    ds = xr.open_dataset(file_name)
    # Extract the 'pr' variable from the dataset using the indices
    # Assuming 'lon_index' and 'lat_index' are valid indices for your dataset
    pr_data = ds['ET'].isel(lon=lon_indices, lat=lat_indices)
    pr_df = pr_data.to_dataframe().reset_index()
    pr_df_merged = pr_df.merge(filtered_df, on=['lat','lon'],how='left')
    pr_df_merged.drop(['lat_index','lon_index'],axis=1,inplace=True)
    mean_pr_df = pr_df_merged.groupby(by=['time','abbre']).mean()
    # Pivot the DataFrame
    pr_df = mean_pr_df.reset_index().pivot(index='time', columns='abbre', values='ET')
    
    # Optional: Rename columns to make them more descriptive (if needed)
    pr_df.columns.name = None  # Remove the name of the columns index if undesired
    
    # Convert cftime.DatetimeNoLeap to pandas datetime
    pr_df.index = pr_df.index.map(lambda x: x.strftime('%Y-%m-%d') if hasattr(x, 'strftime') else x)
    pr_df.index = pd.to_datetime(pr_df.index)
    
    # Format time index as year-month
    pr_df.index = pr_df.index.to_period('D')
    # Display the resulting DataFrame
    pr_df.to_csv(os.path.join('zonal_mean_df'+file_name[:4]+'.csv'))
    print(file_name[:4]+'.csv')


2012.csv
2011\ET.SiTHv2.A2011.nc
2011.csv
2010\ET.SiTHv2.A2010.nc
2010.csv
2009\ET.SiTHv2.A2009.nc
2009.csv
2008\ET.SiTHv2.A2008.nc
2008.csv
2007\ET.SiTHv2.A2007.nc
2007.csv
2006\ET.SiTHv2.A2006.nc
2006.csv
2005\ET.SiTHv2.A2005.nc
2005.csv
2004\ET.SiTHv2.A2004.nc
2004.csv
2003\ET.SiTHv2.A2003.nc
2003.csv
2002\ET.SiTHv2.A2002.nc
2002.csv
2001\ET.SiTHv2.A2001.nc
2001.csv
2000\ET.SiTHv2.A2000.nc
2000.csv
